In [ ]:
# === 1. CONFIGURATION ===
reg = {
    'dataset': 'mlc25',      # Choices: 'monk1', 'mlc25', 'mnist', 'fmnist', 'kmnist'
    'model': 'svr',          # Choices: 'svc', 'svr'
    'svm_kernel': 'rbf',     # Choices: 'rbf', 'linear', 'poly', 'sigmoid'
    'C': 1.0,                # Regularization parameter C (float)
    'gamma': 'scale',        # Kernel coefficient gamma (float or 'scale'/'auto')
    'BATCH_SIZE': 1024,      # Large batch size since all data is extracted anyway
    'data_root': './data'
}

cls = {
    'dataset': 'monk1',      # Choices: 'monk1', 'mlc25', 'mnist', 'fmnist', 'kmnist'
    'model': 'svc',          # Choices: 'svc', 'svr'
    'svm_kernel': 'rbf',     # Choices: 'rbf', 'linear', 'poly', 'sigmoid'
    'C': 1.0,                # Regularization parameter C (float)
    'gamma': 'scale',        # Kernel coefficient gamma (float or 'scale'/'auto')
    'BATCH_SIZE': 1024,      # Large batch size since all data is extracted anyway
    'data_root': './data'
}

In [ ]:
cfg = cls  # Choose between 'reg' and 'cls' configurations

In [ ]:
print("--- Configuration Loaded ---")
for key, value in cfg.items():
    print(f"{key}: {value}")
print("----------------------------")

In [ ]:
# === 2. IMPORTS AND UTILITIES ===
import torch
import numpy as np
import sys
import os
import requests
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

# Import SVM models and metrics
try:
    from models import SVCModel, SVRModel
    from sklearn.metrics import *
    from sklearn.multioutput import MultiOutputRegressor
    # Assuming data_loader.py functions are available/imported
    from utils.data_loader import get_monk1_data, get_ml_cup_data
    
except ImportError as e:
    print("FATAL ERROR: A required module was not found.")
    print(f"Error: {e}")
    # In Jupyter, we raise the error but don't exit the kernel
    
    
# --- Utility function to extract data from DataLoader to NumPy arrays ---
def extract_data_to_numpy(data_loader):
    """
    Converts data from a PyTorch DataLoader into a flattened NumPy array pair (X, y).
    Targets (y) are returned in their original dimensionality (e.g., [N, M] for multi-output).
    """
    X_list = []
    y_list = []
    for X, y in data_loader:
        # Flatten the input (e.g., 28x28 image -> 784 features)
        X_list.append(X.view(X.size(0), -1).numpy()) 
        # Convert labels to NumPy
        y_list.append(y.numpy())
    
    X_data = np.concatenate(X_list)
    y_data = np.concatenate(y_list)
    
    # We return y_data as is (2D array, e.g., [N, 1] or [N, M]).
    return X_data, y_data

In [ ]:
from utils.data_loader import get_monk1_data

# === 3. DATA LOADING AND PREPARATION ===
dataset_name = cfg['dataset']
BATCH_SIZE = 1024
data_root = cfg['data_root']

print(f"Loading dataset: {dataset_name.upper()}...")

# --- 3.1 Load Data Logic (Include all if/elif blocks for monk1, mlc25, mnist, fmnist, kmnist) ---    
print(f"Loading dataset: {dataset_name}...")
    
# Determine task type and load data (DataLoader objects are returned)
if dataset_name == 'monk1':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk1_data(BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"
elif dataset_name == 'mlc25':
    train_loader, validation_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_ml_cup_data(BATCH_SIZE, data_root)
    is_regression_task = True
    metric_name = "Test MEE"
else:
    # This block handles the error if the dataset is outside the specified choices (monk1, mlc25).
    print("Unsupported dataset for SVM.")
    sys.exit(1)


# --- 3.2 Data Preparation for Scikit-learn ---

# Convert DataLoaders (PyTorch) to NumPy arrays (Scikit-learn)
X_train, y_train = extract_data_to_numpy(train_loader)
X_test, y_test = extract_data_to_numpy(test_loader)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")

In [ ]:
# === 4. MODEL INITIALIZATION ===
model_type = cfg['model']
model_kernel = cfg['svm_kernel']
model_C = cfg['C']
model_gamma = cfg['gamma']
is_multi_output = is_regression_task and (y_train.ndim > 1 and y_train.shape[1] > 1)

# Create the base model with hyperparameters
base_svm_model = None
if model_type == 'svc':
    if is_regression_task:
        print("ERROR: SVC (Classification) cannot be used for Regression (mlc25).")
        sys.exit(1)
    # Get the scikit-learn SVC object
    base_svm_model = SVCModel(kernel=model_kernel, C=model_C, gamma=model_gamma).model
        
elif model_type == 'svr':
    # Get the scikit-learn SVR object
    base_svm_model = SVRModel(kernel=model_kernel, C=model_C, gamma=model_gamma).model
    
# --- Assign the Final Model Container (Handle Multi-Output) ---
if is_regression_task and y_train.shape[1] > 1:
    # If Regression AND Multi-Output (MLC25), use the MultiOutputRegressor wrapper
    model_container = MultiOutputRegressor(base_svm_model)
else:
    # If Classification (MONK1) or Single-Output Regression, use the base model
    model_container = base_svm_model

print(f"Using Model: {model_type.upper()} (Kernel: {cfg['svm_kernel']}, Multi-Output: {is_multi_output})")

In [ ]:
def mean_euclidean_error(y_true, y_pred):
    errors = y_true - y_pred
    return np.linalg.norm(errors, axis=1).mean()

In [ ]:
from sklearn.metrics import make_scorer

mee_scorer = make_scorer(mean_euclidean_error, greater_is_better=False, needs_proba=False, needs_threshold=False)

In [ ]:
# === 5. TRAINING AND EVALUATION ===

# 1. Prepare Target Data Format (y) - Dimensionality handling
if is_multi_output:
    y_train_fit = y_train 
    y_test_eval = y_test
else:
    # Classification (SVC) or Single-Output Regression: target 1D [N,]
    y_train_fit = y_train.ravel() 
    y_test_eval = y_test.ravel() 

print("\n--- Starting SVM Training ---")

# 2. Training (model_container is SVC, SVR, or MultiOutputRegressor)
# NOTE: Training time can be significant for large datasets like MNIST!
model_container.fit(X_train, y_train_fit) 

print("--- Training Completed ---")

In [ ]:
# 3. Evaluation
y_test_pred = model_container.predict(X_test)

if is_regression_task:
    # Calculate MEE for Regression
    final_metric = mean_euclidean_error(y_test_eval, y_test_pred)
else:
    # Calculate Accuracy for Classification
    final_metric = accuracy_score(y_test_eval, y_test_pred) * 100.0


In [ ]:
# --- 5. Print Final Results ---
print("\n--- Final Results ---")
print(f"Model: {model_type.upper()} (Kernel: {cfg['svm_kernel']})")
print(f"Dataset: {dataset_name.upper()}")
print(f"{metric_name}: {final_metric:.4f}")
print("---------------------")

In [ ]:
# --- ADDITIONAL VISUALIZATIONS AND METRICS ---

# 1) Classification
if not is_regression_task:
    try:
        y_scores = model_container.decision_function(X_test)
    except AttributeError:
        print("Model does not support decision_function; skipping ROC and AUC calculations.")
        y_scores = y_test_pred

    # Confusion Matrix
    cm = confusion_matrix(y_test_eval, y_test_pred)
    ConfusionMatrixDisplay(cm).plot()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test_eval, y_scores)
    RocCurveDisplay(fpr = fpr, tpr = tpr).plot()

    # AUC Score
    auc_score = roc_auc_score(y_test_eval, y_scores) * 100.0
    print(f"AUC Score (%): {auc_score:.4f}")